# Private AI Studio — 画像から動画

**画像データ入りです。必ずPrivateのまま利用してください。** GPU T4 / Internet ONにしてRun All。モデルの利用条件を確認してください。SVDは文章の動作指示に非対応です。14フレームを生成し、fpsで再生時間を調整します。初回の実生成は未検証です。


In [ ]:
import sys, subprocess
# Keep Kaggle CUDA PyTorch intact. Install only the inference dependencies.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "diffusers==0.35.1", "transformers==4.56.2", "accelerate==1.10.1", "safetensors==0.6.2", "imageio==2.37.0", "imageio-ffmpeg==0.6.0", "Pillow==11.3.0"])


In [ ]:
import json, base64, io, os, time, traceback
from pathlib import Path
from PIL import Image, ImageOps
from IPython.display import display, Video, FileLink
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video

CONFIG = json.loads("{\"schema_version\":1,\"provider\":\"kaggle-svd\",\"preset\":\"light\",\"width\":576,\"height\":320,\"steps\":15,\"model\":\"stabilityai/stable-video-diffusion-img2vid\",\"frames\":14,\"fps\":7,\"motion\":127,\"seed\":42,\"image_base64\":\"iVBORw0KGgoAAAANSUhEUgAAAtAAAAUACAIAAADhrPJqAAAjCUlEQVR4nO3ce5BeZWHH8ZOFBAQpIkVFhxZLvaGCUhu5KrQyCGhFKbZjC44DY4sTK4oIFASvyEUoUkS8UEGxo6IFBKwKhSKgolJE5KZiHUbbqTheqqhckvSP7azLJtm8l/M75znnfD5/hSS8ecnw/Pa7z9lkyda77lcBACTNtP0GAID+ExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIjbsO03ALTvluO3Sv8SO77j3vQvAZRsyda77tf2ewCyGuiJ6SkS6DfBAf3RibAYlxCBfhAc0Em9bIvRqRDoHMEBHTDwvBiFBIHCCQ4okcKYkv6A0ggOKIXICBEfUALBAW0SGQ0TH9AWwQFNExmFEB/QJMEBTRAZhRMfkCY4IEVkdJT4gATBATXTGb2hPKBGggPqoTN6THnA9AQHTEVnDIrygIkJDpiEzhg45QHjEhwwBp3BAsoDRiQ4YP10BuulPGBxggMWIzUYi+yAdREcsBY6gykpD1hAcMDDSA1qJDtgjuCAqtIZhCkPEBwMndSgMbKDIRMcDJfUoBWyg2ESHAyOzqAQyoNBERwMiNSgQLKDgRAcDILUoHCyg94THPSc1KBDZAc9JjjoLalBR8kOeklw0ENSgx6QHfSM4KBXpAY9IzvoDcFBT0gNekx20AOCg86TGgyE7KDTZtp+AzAVtcFw+L+dTnPDQVcZXwbLVQddJDjoHqkBleygawQHXSI1YAHZQVf4Gg46Q23AmpwLusINBx1gUmG9XHVQOMFB0aQGjEV2UCyPVCiX2oBxOTUUyw0HJTKaMCVXHZTGDQfFURswPeeI0rjhoCAmEmrnqoNCuOGgFGoDEpwsCuGGg/YZRGiAqw7a5YaDlqkNaIazRrvccNAa8wetcNVBK9xw0A61AW1x+miFGw6aZuygEK46aJIbDhqlNqAcziNNEhw0x7pBaZxKGuORCk0walA4j1dIc8NBnNqA8jmnpAkOsqwYdIXTSpRHKqQYL+goj1dIcMNBhNqA7nJ+SRAc1M9aQdc5xdTOIxXqZKSgZzxeoS5uOKiN2oD+ca6pi+CgHlYJ+srpphaCgxrYI+g3Z5zpCQ6mZYlgCJx0puSLRpmcAYIB8mWkTMYNBxNSGzBMzj6TERxMwuLAkFkAJiA4GJutAewA4xIcjMfKALOsAWMRHIzBvgDz2QRGJzgYlWUB1mQZGJHgYCQ2BVgX+8Ao/D0crIcpAUbkr+hgEW44WIzaAEZnMViE4GCdbAcwLrvBuggO1s5qAJOxHqyV4GAt7AUwDRvCmgQHC1kKYHqWhAUEBw9jI4C62BPmExz8lnUA6mVVmCM4+H92AUiwLcwSHFSVRQCSLAyV4KCyBUCenUFwDJ0VAJphbQZOcAya8w80yeYMmeAAAOIEx3D5VANonuUZLMExUM480Bb7M0yCY4icdqBdVmiABMfgOOdACWzR0AiOYXHCgXJYpEERHAPibAOlsUvDITiGwqkGymSdBkJwDILzDJTMRg2B4AAA4gRH//nUASifpeo9wdFzzjDQFfaq3wRHnzm9QLdYrR4THL3l3AJdZLv6SnD0kxMLdJcF6yXBAQDECY4e8skB0HV2rH8ER984pUA/WLOeERy94nwCfWLT+kRwAABxgqM/fCoA9I9l6w3B0RPOJNBX9q0fBEcfOI1Av1m5HhAcAECc4Og84Q8Mga3rOsHRbU4gMBwWr9MER4c5e8DQ2L3uEhwAQJzg6CqZDwyT9esowdFJzhswZDawiwQHABAnOLpH2gNYws4RHB3jjAHMsofdIjgAgDjB0SVyHmA+q9ghgqMznCuANdnGrhAcAECc4OgGCQ+wLhayEwRHBzhLAIuzk+UTHABAnOAonWwHGIW1LJzgKJrzAzA6m1kywQEAxAmOckl1gHFZzmIJDgAgTnAUSqQDTMZ+lklwlMhpAZiGFS2Q4AAA4gRHcYQ5wPRsaWkEBwAQJzjKIskB6mJRiyI4AIA4wVEQMQ5QL7taDsFRCqcCIMG6FkJwAABxgqMIAhwgx8aWQHAAAHGCo33SGyDN0rZOcAAAcYKjZaIboBn2tl2CAwCIExxtktsATbK6LRIcAECc4GiN0AZonu1ti+AAAOIERzskNkBbLHArBAcAECc4WiCuAdplh5snOACAOMHRNFkNUAJr3DDBAQDECQ4AIE5wNMoNHkA5bHKTBAcAECc4miOlAUpjmRsjOACAOMHREBENUCb73AzBAQDECQ4AIE5wNMF9HUDJrHQDBAcAECc44oQzQPlsdZrgAADiBAcAECc4stzRAXSFxY4SHABAnOAIEssA3WK3cwQHABAnOACAOMGR4l4OoIusd4jgAADiBAcAECc4ItzIAXSXDU8QHABAnOAAAOIER/3cxQF0nSWvneAAAOIEBwAQJzhq5hYOoB/seb0EBwAQJzgAgDjBUSf3bwB9YtVrJDgAgDjBAQDECY7auHkD6B/bXhfBAQDECQ4AIE5wAABxgqMeHvIB9JWFr4XgAADiBAcAECc4AIA4wVEDj/cA+s3OT09wAABxggMAiBMcAECc4JiWB3sAQ2DtpyQ4AIA4wQEAxAkOACBOcEzFIz2A4bD50xAcAECc4AAA4gQHABAnOCbnYR7A0Fj+iQkOACBOcAAAcYIDAIgTHABAnOCYkK8bAhgm+z8ZwQEAxAkOACBOcAAAcYIDAIgTHABAnOCYhC9RBhgyHwUmIDgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4BibPw0FgI8F4xIcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCYzz+4DUAs3xEGIvgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzjG4G+xBWA+HxdGJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOMaw4zvubfstAFAQHxdGJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJjvH4W2wBmOUjwlgEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4BibP3gNgI8F4xIcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnBMwp+GAhgyHwUmIDgAgDjBAQDECQ4AIE5wAABxggMAiBMcE/IlygDDZP8nIzgAgDjBAQDECQ4AIE5wAABxgmNyvm4IYGgs/8QEBwAQJzgAgDjBAQDECY6peJgHMBw2fxqCAwCIExwAQJzgAADiBMe0PNIDGAJrPyXBAQDECQ4AIE5wAABxgqMGHuwB9Judn57gAADiBAcAECc4AIA4wVEPj/cA+srC10JwAABxggMAiBMcAECc4KiNh3wA/WPb6yI4AIA4wQEAxAmOOrl5A+gTq14jwQEAxAkOACBOcNTM/RtAP9jzegkOACBOcAAAcYKjfm7hALrOktdOcAAAcYIDAIgTHBHu4gC6y4YnCA4AIE5wAABxgiPFjRxAF1nvEMEBAMQJDgAgTnAEuZcD6Ba7nSM4AIA4wZEllgG6wmJHCQ4AIE5wAABxgiPOHR1A+Wx1muAAAOIERxOEM0DJrHQDBAcAECc4AIA4wdEQ93UAZbLPzRAcAECc4GiOiAYojWVujOAAAOIER6OkNEA5bHKTBAcAECc4AIA4wdE0N3gAJbDGDRMcAECc4GiBrAZolx1unuAAAOIERzvENUBbLHArBAcAECc4WiOxAZpne9siOACAOMHRJqEN0CSr2yLBAQDECY6WyW2AZtjbdgkOACBOcLRPdAOkWdrWCQ4AIE5wFEF6A+TY2BIIDgAgTnCUQoADJFjXQgiOgjgVAPWyq+UQHABAnOAoixgHqItFLYrgAADiBEdxJDnA9GxpaQQHABAnOEokzAGmYUULJDgK5bQATMZ+lklwAABxgqNcIh1gXJazWIIDAIgTHEWT6gCjs5klExylc34ARmEtCyc4AIA4wdEBsh1gcXayfIKjG5wlgHWxkJ0gOACAOMHRGRIeYE22sSsER5c4VwDzWcUOERwAQJzg6Bg5DzDLHnaL4OgeZwzAEnaO4AAA4gRHJ0l7YMhsYBcJjq5y3oBhsn4dJTgAgDjB0WEyHxgau9ddgqPbnD1gOCxepwmOznMCgSGwdV0nOACAOMHRB8If6Dcr1wOCoyecRqCv7Fs/CI7+cCaB/rFsvSE4AIA4wdErPhUA+sSm9Yng6BvnE+gHa9YzgqOHnFKg6+xY/wgOACBOcPSTTw6A7rJgvSQ4esuJBbrIdvWV4Ogz5xboFqvVY4Kj55xeoCvsVb8Jjv5zhoHyWareExwAQJzgGASfOgAls1FDIDiGwnkGymSdBkJwDIhTDZTGLg2H4BgWZxsoh0UaFMExOE44UAJbNDSCY4icc6BdVmiABMdAOe1AW+zPMAmO4XLmgeZZnsESHABAnOAYNJ9qAE2yOUMmOIbO+QeaYW0GTnBgBYA4O4PgoKpsAZBkYagEB3MsApBgW5glOPgtuwDUy6owR3DwMNYBqIs9YT7BwUI2ApieJWEBwcFaWApgGjaENQkO1s5eAJOxHqyV4GCdrAYwLrvBuggOFmM7gNFZDBaxZOtd92v7PdABtxy/VdtvASiX1GC93HAwEmsCrIt9YBSCg1HZFGBNloERCQ7GYFmA+WwCoxMcjMe+ALOsAWMRHIzNygB2gHEJDiZha2DILAATEBxMyOLAMDn7TMbfw8G0/BUdMBBSg2m44WBaNgiGwElnSoKDGlgi6DdnnOkJDuphj6CvnG5qITiojVWC/nGuqYsvGqV+vowUekBqUC83HNTPTkHXOcXUTnAQYa2gu5xfEjxSIcvjFegQqUGOGw6y7Bd0hdNKlOAgzopB+ZxT0jxSoTker0CBpAbNcMNBc+walMappDGCg0ZZNyiH80iTPFKhHR6vQIukBs1zw0E77B20xemjFW44aJmrDmiM1KBFbjhomQWEZjhrtMsNB6Vw1QEhUoMSuOGgFDYREpwsCuGGg+K46oBaSA2K4oaD4lhJmJ5zRGnccFAuVx0wAalBmdxwUC67CeNyaiiWGw46wFUHrJfUoHCCg86QHbBWUoNO8EiFzrCqsCbngq5ww0H3uOqASmrQNYKDrpIdDJbUoIsEB90mOxgUqUF3+RoOus3+Mhz+b6fT3HDQE6466DGpQQ8IDnpFdtAzUoPeEBz0kOygB6QGPSM46C3ZQUdJDXpJcNBzsoMOkRr0mOBgEGQHhZMa9J7gYEBkBwWSGgyE4GBwZAeFkBoMiuBguJQHrdAZDJPgYOhkB42RGgyZ4ICqkh2ESQ0QHPAwyoMa6QyYIzhgLWQHU5IasIDggMUoD8aiM2BdBAesn+xgvaQGLE5wwBiUBwvoDBiR4IBJKI+B0xkwLsEBU1Eeg6IzYGKCA+qhPHpMZ8D0BAfUTHn0hs6AGgkOSFEeHaUzIEFwQBPER+FEBqQJDmia+CiEyIAmCQ5ok/homMiAtggOKIX4CBEZUALBASUSH1MSGVAawQEdoD/WS2FA4QQHdNLAE0ReQOcIDuiPXlaItoB+EBzQf50IEWEB/SY4gCaKRE/AwAkOACBupu03AAD0n+AAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiNuw7TdA0b539cU3337X6qpausEGJ571wW/c8e25H3rKE39/52c944KLr1jvi6w4+KCzP3rR4j/nFS/e511HvuY5L3vlvT/5WVVVd37+k0/d5+VzP/rMp2x33OGv2nDDDVc+tPL1J535Xz+6d5TXnHud0d9qVVX7Pn+Xww56SVVVy3d8+ldvua2qqg9/+vKz3nzkzbffNfsTvnD9je//+MUL3tKOT/vDNf+ty6+5fpRfEWAIlmy9635tvwfKNfeB/2nbbXvG3x+x76FHTPMiizj/lBPuvueH3/7+PZ+44so1/5WrLjj74KPe8t8/+vH+e+724j/d42/ffPIorzniLz3Kv7vm66z5lqb/FQF6zCMVRnLH3d//va0fV1XVnZ//5D8c9/pDD/qz2W9XVXXVBWc/bqstq6patnTpDR//4NO22/bS9512zYXnvPovDqiq6o2H/fWmmzzi42e+Y/PNHnn2iW/85HveefE5pz57+yfPf/FHbLzRJhtv9M+XfX7v3Zav9VffcovNN1q2rKqqL1x/44c/ddnca869h1mz397q0Y+64NQTLnnfqWce/4YFP/SYLbe48PS3XnzOqRee/tbHbLnFNL8hC97SNC8FMASCg5Hs/pxn3fadu6uq2mjZ0kuvuva8iz4z90OXX3P9PrvvXFXVbn+0wzU33vTKl+1/0rnnv/Q1Rx/+VwdWVfXuD114369+/ZdHHH/CikP/6aLLXv6641a89bTTjv67+S++53N3uubGm+6+5wfbPO6xS5eu5THfyedecMk5p55x7OuW77D9jbfcNveaa32rJ6w47NKrrjvg8Dd97otf3nijZfN/6MQVh11y5bUvfc2bLrny2hNWHDrNb8iCtzTNSwEMga/hYDHLli799NknL1my5Bf33XfkyWdVVbVy1aovfu3m+T/nsquvf/sRf3PBxVfsvdtzL7ny2tu/+72XvOD5e++2fLNNN5n/0/ZcvtO2T9h69tubbLzRBjMzK1etmv3HffbY5RlP+oP999r9sb/76F2e9cwFr19V1Sc+e9XnrvvKC5+3y9te9+rPfvHLp5/3sbW+25mZmaqqdn32M49813uqqrrqhq+uXLlq/k/YdacdXn/SmVVVfebfrjvu8FfNfufRrz5k+Q7bf+iiS//12i8v/vsw++2Tzj3/pm/dOeJbAmCW4GAxDzz44IErjpn/PQ89tHLVqtXzv+fue36wxeabbbbpJs988nbHnfG+j53+tiv+/YbzPnXZIS/df/5P22CDDV7xhhPuf+CBmZkly3d4+lxtbDAzs902T3jBK1dUVbXnc3fae7flC4Jjy0dt/sRtHv/1W+/4xBVXXnXDV6/+6DnzP7rPRkZVVb/zyE2XLd2wqqq5O5IlM0uWLFnysP+eh//TrFM+8JFxfx8Wf0sArMkjFWrw+eu+suLgg26+/a7Vq1fv+NQnXXb1dRstW7bR0qWzPzozMzMzs+Rrt96+7/N3qarqT3Z+zmsP/u2XVf7xDtvf9t3vzX77xltue/7ynRa8+OrVqz/w9mMf/5itqqraYvPNfvg/P5p7zaqqfvHL+57yxN+vqurAffZavXp1VVVfv/WOF+6xc1VV+z5v1wW98aWbvvmivXavqupFe+3+pf/45sT/vWt9SwAswg0HNbjs6uuv/sh7D3ztMVVVnf8vl3/m/e++/Tv/+fNf/nLZ0qUPPPjgjbd86/xTTjzmtPeedvRrDzlgv4dWrjzq5LPm/t199tj5hptumf32r39z/49/+rMnbbvN/Bf/yc//96hT/vED7zz2N/c/sGrlyjecdGZVVbOvechRbzn+zHPf/45jfvzTn3/j9rvuf+DBqqrectYH3/PmI1/15y/++q13zH7PnLe997wzjj3i4AP2/dWvfzP7OiOa/0jlpm/dedK556/5lgBYhD8WCwDEeaQCAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcYIDAIgTHABAnOAAAOIEBwAQJzgAgDjBAQDECQ4AIE5wAABxggMAiBMcAECc4AAA4gQHABAnOACAOMEBAMQJDgAgTnAAAHGCAwCIExwAQJzgAADiBAcAECc4AIA4wQEAxAkOACBOcAAAcf8HQ6cOE8mE6FQAAAAASUVORK5CYII=\"}")
if not torch.cuda.is_available():
    raise RuntimeError("GPUがありません。Settings → Accelerator → GPU T4に変更してください。")
print("GPU:", torch.cuda.get_device_name(0))
OUTPUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
image = Image.open(io.BytesIO(base64.b64decode(CONFIG.pop("image_base64"), validate=True)))
image = ImageOps.exif_transpose(image).convert("RGB")
image = ImageOps.pad(image, (CONFIG["width"], CONFIG["height"]), color=(16, 16, 16), method=Image.Resampling.LANCZOS)
display(image)
print("入力画像の読み込み完了:", image.size)


In [ ]:
started = time.time()
try:
    pipe = StableVideoDiffusionPipeline.from_pretrained(
        CONFIG["model"], torch_dtype=torch.float16, variant="fp16", low_cpu_mem_usage=True
    )
    pipe.enable_model_cpu_offload()
    pipe.unet.enable_forward_chunking()
    generator = torch.Generator(device="cpu").manual_seed(CONFIG["seed"])
    with torch.inference_mode():
        frames = pipe(
            image, width=CONFIG["width"], height=CONFIG["height"], num_frames=CONFIG["frames"], num_inference_steps=CONFIG["steps"],
            decode_chunk_size=1, generator=generator,
            motion_bucket_id=CONFIG["motion"], noise_aug_strength=0.02,
            fps=7
        ).frames[0]
    output_path = OUTPUT / "private-ai-studio.mp4"
    export_to_video(frames, str(output_path), fps=CONFIG["fps"])
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError("MP4ファイルが作成されませんでした。")
    report = {**CONFIG, "status":"completed", "seconds":round(time.time()-started, 1), "gpu":torch.cuda.get_device_name(0), "torch":torch.__version__}
    (OUTPUT / "generation-report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
    print("生成完了", report)
    display(Video(str(output_path), embed=True))
    display(FileLink(str(output_path)))
except Exception as exc:
    report = {"status":"failed", "error":str(exc), "traceback":traceback.format_exc()}
    (OUTPUT / "generation-error.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
    print("生成失敗。GPU設定・Internet・モデルアクセス権・メモリを確認してください。")
    display(FileLink(str(OUTPUT / "generation-error.json")))
    raise
finally:
    if "pipe" in globals():
        del pipe
    import gc
    gc.collect()
    torch.cuda.empty_cache()
